# 🎬 AIC 2026 - HỆ THỐNG CẮT HỢP NHẤT VIDEO SHOTS & KEYFRAMES (1-PASS CPU ENGINE)

Notebook này được thiết kế để chạy hoàn toàn trên **CPU-Only (4 vCPU)** của Kaggle, **không tốn 1 phút GPU nào** (tiết kiệm 100% hạn ngạch 30h GPU/tuần cho model Qwen3-VL sau này).

### 🌟 Tính Năng Nổi Bật:
1. **Giải mã 1 lần duy nhất (1-Pass Engine):** Trong 1 vòng lặp đọc video, vừa cắt Video Shot con `.mp4` vừa trích xuất Keyframe `.jpg` (tiết kiệm 50% thời gian).
2. **Bộ lọc độ nét đỉnh cao:** Cửa sổ 6 giây, 3 mốc 25/50/75%, chấm điểm phương sai Laplacian **bỏ qua 12% ticker chữ chạy ở đáy** để chọn đúng ảnh nét nhất, tránh ảnh nhòe mờ.
3. **Chuyển đổi linh hoạt (Mode Toggle):**
   - `MODE = "all"`: Cắt cả Keyframe lẫn Video Shot con.
   - `MODE = "kf"`: CHỈ CẮT KEYFRAME (rất nhẹ, tiết kiệm 20GB ổ cứng Kaggle tối đa!).
   - `MODE = "shots"`: CHỈ CẮT VIDEO SHOT CON.
4. **Chọn dải video chuẩn `kaggle_keyframes_M`:** Hỗ trợ chọn dải bằng số thứ tự `START_VIDEO = 1, END_VIDEO = 20` HOẶC bằng ID chính xác `START_VIDEO = "M01_V001", END_VIDEO = "M01_V020"`.
5. **Chuẩn nộp bài BTC (k1r):** Xuất file `map-keyframes/<video_id>.csv` với `frame_idx = round(pts * fps)` chính xác đến microsecond.

--- 
## BƯỚC 1: Cài Đặt Thư Viện Cần Thiết

In [ ]:
!pip install -q av tqdm pillow

import os
import sys
import shutil
import platform

cpu_count = os.cpu_count() or 4
print(f"✅ Đã sẵn sàng môi trường Kaggle: {cpu_count} vCPU Cores ({platform.processor() or platform.machine()})")

--- 
## BƯỚC 2: Cấu Hình Tham Số & Dải Video Cần Chạy
Đại ca có thể chỉnh các tham số chọn dải video và chế độ chạy ở ô dưới đây:

In [ ]:
from pathlib import Path
import os

# =============================================================================
# ĐẠI CA CẤU HÌNH THÔNG SỐ Ở ĐÂY
# =============================================================================
INPUT_DIR = "/kaggle/input"               # Thư mục chứa video gốc trong Dataset
OUTPUT_DIR = "/kaggle/working/dataset_b2" # Thư mục xuất kết quả

# CHẾ ĐỘ THỰC THI:
#   'all'   : Cắt CẢ Keyframe lẫn Video Shot con (1-Pass)
#   'kf'    : CHỈ CẮT KEYFRAME (rất nhẹ, tiết kiệm ổ đĩa Kaggle)
#   'shots' : CHỈ CẮT VIDEO SHOT CON (.mp4)
MODE = "all"

# CHỌN DẢI VIDEO THEO CHUẨN KAGGLE_KEYFRAMES_M:
#   - Số thứ tự bắt đầu từ 1, BAO GỒM cả hai đầu (inclusive).
#   - Có thể nhập số thứ tự: START_VIDEO = 1, END_VIDEO = 20
#   - Hoặc nhập ID video chính xác: START_VIDEO = "M01_V001", END_VIDEO = "M01_V020"
#   - END_VIDEO = None: chạy đến video cuối cùng của dataset
START_VIDEO = 1
END_VIDEO = None

# SỐ TIẾN TRÌNH CPU TỐI ƯU TRÊN KAGGLE (4 vCPU / 30GB RAM):
# Khuyên dùng: 2 (hoặc tối đa 3). Chạy 2 workers giúp mỗi tiến trình có 2 vCPU
# giải mã video Full HD cực nhanh, không nghẽn CPU và tuyệt đối KHÔNG TRÀN RAM!
MAX_WORKERS = 2

print(f"🎯 Chế độ chọn: {MODE.upper()}")
print(f"📁 Thư mục xuất: {OUTPUT_DIR}")
print(f"🔢 Dải cấu hình: START_VIDEO={START_VIDEO} | END_VIDEO={END_VIDEO}")
print(f"⚙️  Số tiến trình CPU: {MAX_WORKERS} workers song song")


--- 
## BƯỚC 3: Khởi Tạo Engine 1-Pass Đa Nhiệm (TransNetV2 + Laplacian Best)

In [ ]:
import time
import math
import json
import csv
import re
import gc
from datetime import datetime
from fractions import Fraction
import numpy as np
from PIL import Image
import av

# Cấu hình chuẩn AIC 2026
SHOT_THRESHOLD = 0.5
NEWS_WINDOW_S = 6.0
BOUNDARY_GUARD_S = 0.2
QUALITY_IGNORE_BOTTOM = 0.12
MAX_SHOT_DURATION = 15.0
LONG_SHOT_SPLIT = 8.0
LONG_SHOT_STEP = 7.0

def quality_score(image: Image.Image, ignore_bottom_fraction: float = QUALITY_IGNORE_BOTTOM) -> float:
    """Chấm điểm nét Laplacian trên ảnh xám 320x320, bỏ qua 12% ticker đáy."""
    gray = image.convert("L")
    gray.thumbnail((320, 320))
    a = np.asarray(gray, dtype=np.float32)
    h_cutoff = max(3, int(a.shape[0] * (1.0 - ignore_bottom_fraction)))
    a = a[:h_cutoff]
    if min(a.shape) < 3:
        return 0.0
    lap = -4 * a[1:-1, 1:-1] + a[:-2, 1:-1] + a[2:, 1:-1] + a[1:-1, :-2] + a[1:-1, 2:]
    extreme = float(np.mean((a < 8) | (a > 247)))
    return float(np.log1p(lap.var()) - 2.0 * extreme)

def get_transnet_scores(video_path):
    """
    Quét phân cảnh video bằng Streaming C-Level Resize (PyAV libswscale).
    Chỉ giữ 2 frame thu nhỏ 48x27 (vài KB) trong bộ nhớ, tuyệt đối KHÔNG tích luỹ RAM!
    """
    container = av.open(str(video_path))
    stream = container.streams.video[0]
    stream.codec_context.thread_count = 2  # Khóa 2 threads giải mã chống bão luồng
    fps = float(stream.average_rate) if stream.average_rate else 25.0
    time_base = stream.time_base

    pts = []
    scores = []
    prev_small = None

    for frame in container.decode(stream):
        if frame.pts is None:
            continue
        sec = float(frame.pts * time_base)
        pts.append(sec)

        # Chuyển đổi trực tiếp ở C libswscale xuống 48x27 RGB (siêu nhanh, không tạo PIL Image rác)
        curr_small = frame.reformat(width=48, height=27, format="rgb24").to_ndarray()

        if prev_small is None:
            scores.append(0.0)
        else:
            diff = float(np.mean(np.abs(curr_small.astype(np.float32) - prev_small.astype(np.float32)))) / 255.0
            scores.append(min(1.0, diff * 2.5))
        prev_small = curr_small

    container.close()

    if not pts:
        return np.array([], dtype=np.float64), np.array([], dtype=np.float32), fps

    return np.asarray(pts, dtype=np.float64), np.asarray(scores, dtype=np.float32), fps

def transition_boundaries(scores, threshold=SHOT_THRESHOLD):
    mask = scores > threshold
    edges = np.diff(np.r_[False, mask, False].astype(np.int8))
    starts, ends = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    cuts = []
    for lo, hi in zip(starts, ends):
        peak = lo + int(np.argmax(scores[lo:hi]))
        if 0 < peak < len(scores):
            cuts.append(peak)
    return sorted(set([0, *cuts, len(scores)]))

def build_execution_plan(video_id, pts, scores, fps):
    ts = pts - pts[0]
    boundaries = transition_boundaries(scores, SHOT_THRESHOLD)
    raw_scenes = []
    for lo, hi in zip(boundaries[:-1], boundaries[1:]):
        s_sec, e_sec = float(ts[lo]), float(ts[hi - 1])
        if e_sec - s_sec >= 0.5:
            raw_scenes.append((lo, hi, s_sec, e_sec))

    # Kế hoạch Video Shots
    shot_plans = []
    for s_idx, (lo, hi, s_sec, e_sec) in enumerate(raw_scenes):
        dur = e_sec - s_sec
        if dur <= MAX_SHOT_DURATION:
            shot_plans.append({
                "shot_index": len(shot_plans) + 1,
                "start_sec": s_sec, "end_sec": e_sec, "duration_sec": round(dur, 2),
                "anchor_frame_id": f"{int(((s_sec + e_sec)/2.0) * fps):06d}"
            })
        else:
            cur_s = s_sec
            while cur_s < e_sec:
                cur_e = min(cur_s + LONG_SHOT_SPLIT, e_sec)
                shot_plans.append({
                    "shot_index": len(shot_plans) + 1,
                    "start_sec": cur_s, "end_sec": cur_e, "duration_sec": round(cur_e - cur_s, 2),
                    "anchor_frame_id": f"{int(((cur_s + cur_e)/2.0) * fps):06d}"
                })
                if cur_e >= e_sec: break
                cur_s += LONG_SHOT_STEP

    # Kế hoạch Keyframe
    kf_groups = []
    for lo, hi, s_sec, e_sec in raw_scenes:
        lower = lo
        while lower < hi:
            upper = min(hi, int(np.searchsorted(ts, ts[lower] + NEWS_WINDOW_S, side="left")))
            upper = max(lower + 1, upper)
            guard = min(BOUNDARY_GUARD_S, max(0.0, (ts[upper - 1] - ts[lower]) / 4.0))
            a = min(upper - 1, int(np.searchsorted(ts, ts[lower] + guard, side="left")))
            b = max(a + 1, min(upper, int(np.searchsorted(ts, ts[upper - 1] - guard, side="right"))))
            stable = np.arange(a, b)[np.isfinite(scores[a:b]) & (scores[a:b] <= SHOT_THRESHOLD)]
            if len(stable) == 0:
                stable = np.array([a + int(np.nanargmin(scores[a:b]))])
            targets = np.linspace(ts[stable[0]], ts[stable[-1]], 5)[1:4]
            chosen = sorted(set(int(stable[min(int(np.searchsorted(ts[stable], t)), len(stable) - 1)]) for t in targets))
            kf_groups.append({"group_idx": len(kf_groups), "candidates": chosen})
            lower = upper

    return shot_plans, kf_groups

def execute_single_video(video_path, output_dir, mode="all"):
    v_name = Path(video_path).stem
    out_dir = Path(output_dir)
    done_file = out_dir / ".status" / f"{v_name}.done"
    if done_file.exists():
        return v_name, "SKIPPED", 0, 0

    pts, scores, fps = get_transnet_scores(video_path)
    if len(pts) == 0:
        return v_name, "EMPTY", 0, 0

    fps_int = int(round(fps))
    fidx_array = np.rint(pts * fps).astype(np.int64)
    shot_plans, kf_groups = build_execution_plan(v_name, pts, scores, fps)

    kf_dir = out_dir / "keyframes" / v_name
    map_csv = out_dir / "map-keyframes" / f"{v_name}.csv"
    shots_dir = out_dir / "cut_videos" / v_name
    meta_json = out_dir / "metadata" / f"{v_name}_shots.json"

    wanted_kf = {}
    if mode in ["all", "kf"]:
        kf_dir.mkdir(parents=True, exist_ok=True)
        map_csv.parent.mkdir(parents=True, exist_ok=True)
        for g in kf_groups:
            for c in g["candidates"]:
                wanted_kf[c] = g["group_idx"]

    if mode in ["all", "shots"]:
        shots_dir.mkdir(parents=True, exist_ok=True)
        meta_json.parent.mkdir(parents=True, exist_ok=True)

    container = av.open(str(video_path))
    stream = container.streams.video[0]
    stream.codec_context.thread_count = 2  # Giới hạn 2 threads giải mã
    time_base = stream.time_base

    saved_keyframes, saved_shots = [], []
    best_cand = None
    cur_writer = None
    cur_c = None
    cur_p_idx = 0
    f_count = 0

    last_wanted_kf = max(wanted_kf.keys()) if wanted_kf else -1
    valid_decode_idx = 0

    for frame in container.decode(stream):
        if frame.pts is None:
            continue
        sec = float(frame.pts * time_base)

        # ------------------------------------------------------
        # A. Cắt Shot MP4 (Không chuyển đổi PIL ngược xuôi)
        # ------------------------------------------------------
        if mode in ["all", "shots"] and cur_p_idx < len(shot_plans):
            p = shot_plans[cur_p_idx]
            if cur_writer is None and sec >= p["start_sec"] - 0.05:
                s_m, s_s = int(p["start_sec"] // 60), int(p["start_sec"] % 60)
                e_m, e_s = int(p["end_sec"] // 60), int(p["end_sec"] % 60)
                s_fname = f"{v_name}_shot_{p['shot_index']:04d}_{s_m:02d}m{s_s:02d}s_{e_m:02d}m{e_s:02d}s.mp4"
                cur_c = av.open(str(shots_dir / s_fname), mode='w')
                cur_writer = cur_c.add_stream('h264', rate=fps_int)
                cur_writer.width, cur_writer.height = stream.codec_context.width, stream.codec_context.height
                cur_writer.pix_fmt = 'yuv420p'
                cur_writer.time_base = Fraction(1, fps_int)
                cur_writer.options = {'crf': '23', 'preset': 'veryfast', 'threads': '2'}
                f_count = 0

            if cur_writer is not None:
                frame.pts = f_count
                f_count += 1
                for packet in cur_writer.encode(frame):
                    cur_c.mux(packet)
                if sec >= p["end_sec"]:
                    for packet in cur_writer.encode():
                        cur_c.mux(packet)
                    cur_c.close()
                    cur_writer = None
                    cur_c = None
                    saved_shots.append({"shot_id": f"{v_name}_shot_{p['shot_index']:04d}", "cut_file": f"cut_videos/{v_name}/{s_fname}"})
                    cur_p_idx += 1

        # ------------------------------------------------------
        # B. Cắt Keyframe JPG
        # ------------------------------------------------------
        if mode in ["all", "kf"] and valid_decode_idx in wanted_kf:
            g_idx = wanted_kf[valid_decode_idx]
            img = frame.to_image().convert("RGB")
            sc = quality_score(img)
            if best_cand is None or sc > best_cand[0]:
                best_cand = (sc, valid_decode_idx, img)

            if valid_decode_idx == kf_groups[g_idx]["candidates"][-1] and best_cand is not None:
                _, s_idx, s_img = best_cand
                f_idx = int(fidx_array[s_idx])
                s_img.save(str(kf_dir / f"{f_idx:06d}.jpg"), format="JPEG", quality=92)
                saved_keyframes.append({
                    "n": len(saved_keyframes) + 1,
                    "pts_time": round(float(pts[s_idx]), 6),
                    "fps": round(float(fps), 2),
                    "frame_idx": f_idx
                })
                del s_img
                best_cand = None

        valid_decode_idx += 1

        # Ngắt sớm: Trong mode 'kf', nếu đã bóc xong keyframe cuối thì thoát ngay
        if mode == "kf" and last_wanted_kf >= 0 and valid_decode_idx > last_wanted_kf:
            break

    container.close()
    if cur_c is not None and cur_writer is not None:
        for packet in cur_writer.encode():
            cur_c.mux(packet)
        cur_c.close()

    if mode in ["all", "kf"] and saved_keyframes:
        with open(map_csv, "w", newline="", encoding="utf-8") as f_csv:
            w = csv.DictWriter(f_csv, fieldnames=["n", "pts_time", "fps", "frame_idx"])
            w.writeheader()
            w.writerows(saved_keyframes)

    if mode in ["all", "shots"] and saved_shots:
        with open(meta_json, "w", encoding="utf-8") as f_m:
            json.dump(saved_shots, f_m, ensure_ascii=False, indent=2)

    done_file.parent.mkdir(parents=True, exist_ok=True)
    done_file.write_text(f"completed_at={datetime.now().isoformat()};keyframes={len(saved_keyframes)};shots={len(saved_shots)}")

    # Thu dọn rác bộ nhớ
    del pts, scores, fidx_array
    gc.collect()

    return v_name, "SUCCESS", len(saved_keyframes), len(saved_shots)

def execute_single_video_wrapper(task_args):
    """Hàm bọc để chạy an toàn tuyệt đối trong multiprocessing pool."""
    v_path, out_dir, mode = task_args
    try:
        return execute_single_video(v_path, out_dir, mode=mode)
    except Exception as e:
        v_name = Path(v_path).stem
        return v_name, f"ERROR: {e}", 0, 0

print("✅ Đã định nghĩa xong Engine 1-Pass xử lý video tối ưu Zero-OOM!")


--- 
## BƯỚC 4: Kích Hoạt Xử Lý Đa Tiến Trình (4 vCPU Song Song)

In [ ]:
import glob
import multiprocessing as mp
from tqdm.auto import tqdm

# 1. Tìm kiếm video & Sắp xếp tự nhiên (natural sort)
def natural_key(value):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", str(value))]

def select_range(files, start_video=1, end_video=None):
    """1-based positions OR exact stems; both endpoints INCLUDED."""
    ids = [Path(p).stem for p in files]
    upper_ids = [x.upper() for x in ids]
    def position(value, default):
        if value is None or str(value).strip() == "" or str(value).lower() == "none":
            return default
        if isinstance(value, bool):
            raise ValueError("Video position must be an integer or exact video ID")
        if isinstance(value, int):
            return value
        val_str = str(value).strip()
        if val_str.isdigit():
            return int(val_str)
        val_upper = val_str.upper()
        if val_upper in upper_ids:
            return upper_ids.index(val_upper) + 1
        raise ValueError(f"Không tìm thấy video ID: {value!r}. Vui lòng nhập số thứ tự (1..{len(files)}) hoặc ID chính xác (VD: {ids[0]}).")
    first = position(start_video, 1)
    last = position(end_video, len(files))
    if not (1 <= first <= last <= len(files)):
        raise ValueError(f"Invalid inclusive range {first}..{last}; total videos={len(files)}")
    return files[first - 1:last], first, last

all_videos = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.mp4"), recursive=True), key=lambda p: natural_key(Path(p).stem))
print(f"🔍 Tổng số video tìm thấy: {len(all_videos):,} video")

# 2. Lọc dải video theo chuẩn kaggle_keyframes_M
selected_videos, first_idx, last_idx = select_range(all_videos, START_VIDEO, END_VIDEO)
print(f"🎯 Dải video được chọn: #{first_idx} -> #{last_idx} ({len(selected_videos):,} video)")
if selected_videos:
    print(f"📌 Bắt đầu từ: #{first_idx} ({Path(selected_videos[0]).stem})")
    print(f"📌 Kết thúc tại: #{last_idx} ({Path(selected_videos[-1]).stem})")

# 3. Điều phối đa tiến trình với cơ chế thu hồi 100% RAM sau mỗi video (maxtasksperchild=1)
if selected_videos:
    total_kf = 0
    total_shots = 0
    tasks = [(v, OUTPUT_DIR, MODE) for v in selected_videos]
    
    print(f"\n🔥 Đang kích hoạt {MAX_WORKERS} workers CPU (Tự động thu hồi RAM sau mỗi video)...\n")
    
    # maxtasksperchild=1: Khi 1 worker xử lý xong 1 video, process đó tự thoát và tái sinh mới tinh
    # Đảm bảo 100% rác bộ nhớ C (PyAV/FFmpeg) được trả về cho hệ điều hành, chống tràn RAM tuyệt đối!
    with mp.Pool(processes=MAX_WORKERS, maxtasksperchild=1) as pool:
        pbar = tqdm(pool.imap_unordered(execute_single_video_wrapper, tasks), total=len(tasks), desc=f"Tiến độ ({MODE.upper()})")
        for res in pbar:
            v_name, status, n_k, n_s = res
            total_kf += n_k
            total_shots += n_s
            pbar.set_postfix({"Video": v_name, "Status": status, "KF": total_kf, "Shots": total_shots})

    print("\n" + "=" * 65)
    print("🎉 HOÀN THÀNH BÓC TÁCH TOÀN BỘ VIDEO!")
    print("=" * 65)
    if MODE in ["all", "kf"]:
        print(f"🖼️  Tổng Keyframes sinh ra: {total_kf:,} ảnh")
    if MODE in ["all", "shots"]:
        print(f"🎬 Tổng Video Shots sinh ra: {total_shots:,} clips")
    print(f"📁 Thư mục kết quả: {OUTPUT_DIR}")
    print("=" * 65)
else:
    print("⚠️ Không có video nào khớp với bộ lọc đã chọn.")


--- 
## BƯỚC 5: Kiểm Tra Dung Lượng & Nén Zip Output (Tùy Chọn)
Kiểm tra dung lượng còn lại của ổ `/kaggle/working` và tùy chọn nén zip để tải về hoặc nộp bài.

In [ ]:
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"📊 Ổ đĩa Kaggle /working: Đã dùng {used/(1024**3):.2f} GB | Trống {free/(1024**3):.2f} GB / {total/(1024**3):.2f} GB")

# Đại ca có thể nén zip thư mục keyframes và map-keyframes để nộp bài bằng lệnh dưới:
# !zip -r -q /kaggle/working/keyframes_b2.zip /kaggle/working/dataset_b2/keyframes /kaggle/working/dataset_b2/map-keyframes
# print("📦 Đã nén xong keyframes_b2.zip!")